# Exploration Doc

### This will be used to test different strategies to optimise parameters

We will look solely at ticker **MSFT**

## Strategy 1
### Show default conditions

Default Parameters: 
| Strategy |Parameter | Value |
|------ | ------ | ------- |
| Moving Average Crossover | Short Window | 50 |
| Moving Average Crossover | Long Window | 200 |
| Momentum | lookback | 20 |
| Mean Reversion | lookback | 20 |
| Mean Reversion | num_std | 2 |
| RSI | window | 20 |
| RSI | overbought | 70 |
| RSI | oversold | 30 |
| Bollinger Band | lookback | 20 |
| Bollinger Band | num_std | 2 |
| MACD | Short Window | 12 |
| MACD | Long Window | 26 |
| MACD | macd_span | 9 |
| Volatility Breakout | lookback | 14 |
| Volatility Breakout | multiplier | 1.5 |


Import all packages

In [ ]:
from src.data import DataLoader


In [ ]:
from src.backtest import BackTester
from src.strategies import *
from src.metrics import metric_summary
from src.visualise import plot_equity_curve, plot_drawdown
from tests.test_metrics import *
import pandas as pd

Get the price history from 2020 to 2025

In [ ]:
ticker = "MSFT"

loader = DataLoader()
data = loader.get_price_history(ticker = ticker,start = "2020-01-01",end = "2025-12-31")

Backtest all the strategies

In [ ]:
backtester = BackTester(starting_capital=1000)

strategies = {
    "Moving Average Crossover": MovingAverageCrossover(),
    "Momentum": MomentumStrategy(),
    "Mean Reversion": MeanReversionStrategy(),
    "RSI": RSIStrategy(),
    "Bollinger Band": BollingerBandBreakoutStrategy(),
    "MACD": MACDStrategy(),
    "Volatility Breakout": VolatilityBreakoutStrategy()
}

results = {}
for name, strategy in strategies.items():
    result = backtester.run_backtest(data=data,strategy=strategy)
    results[name] = metric_summary(result)
    print(f"Finished: {name}")


Print the comparison table

In [ ]:
comparison = pd.DataFrame(results).T
print("\n--- Strategy Comparison ---")
print(comparison.round(4))
comparison.to_csv(f"outputs/comparison_results.csv")



In [ ]:
plot_equity_curve(result, data, title="Mean Reversion vs Buy & Hold")
plot_drawdown(result)

## Strategy 2

### Optimise Sharpe Ratio for momentum

Parameter: **lookback** period

Goal: Find the lookback period that optimises sharpe ratio 

In [ ]:
max_ratio = 0
optimal_period = 0
for period in range(200):
    results = {}
    result = backtester.run_backtest(data=data,strategy=MomentumStrategy(lookback=period))
    results["Momentum"] = metric_summary(result)
    sharpe_ratio = metric_summary(result)["Sharpe Ratio"]
    
    if sharpe_ratio > max_ratio:
        max_ratio = sharpe_ratio
        optimal_period = period
print(optimal_period,": " ,max_ratio)

    